## working with agents on top of carteirinha extracted database

### this also should incorporate the base workflow style:
input: blob, id -> image, id -> llm ->  output: convenio, plano, nome da pessoa e número da carteirinha


In [1]:
import os
import base64
import sys
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Tuple
from PIL import Image   # noqa: F401
from pydantic import BaseModel as PydanticBaseModel
from pydantic import Field
from pydantic_settings import BaseSettings
import oracledb
import io
import cv2
import fitz
import numpy as np
import boto3
from botocore.config import Config
import json
import re
import time
from tqdm import tqdm
import mariadb
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
#strands agetinc workflow
from strands.models import BedrockModel
from strands import Agent

#dont limit the visualization of all the colluns of a pandas dataframe
pd.set_option("display.max_columns", None)

# Add project root to Python path so we can import from app module
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
# Now we can import from app (after adding to sys.path)
from app.utils.logger import get_logger
# Load environment variables from the project root directory
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

logger = get_logger(name=__name__)

## credentials config

In [2]:
@dataclass
class AppConstants:
    BEDROCK_DEFAULT_MODEL_ID: str = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
    DEFAULT_PROMPTS_DIR: str = "prompts/"
    S3_BUCKET_NAME: str = "agente-ai-carteirinha"
    S3_RESULTS_PREFIX: str = "resultados"
    S3_DEBUG_PREFIX: str = "debug"
    STREAMING: bool = False
    CACHE_PROMPT = "default"
    RETRIES: Dict[str, int] = field(default_factory=lambda: {"max_attempts": 3, "mode": "standard"})
    CONNECTION_TIMEOUT: int = 5
    READ_TIMEOUT: int = 60
    TEMPERATURE: float = 0.05
    TOP_P: float = 0.95

In [3]:
class Settings(BaseSettings):
    """Carrega e valida as configurações a partir de variáveis de ambiente."""

    ORACLE_USER: str
    ORACLE_PASSWORD: str
    ORACLE_DSN: str
    ORACLE_INSTANT_CLIENT_PATH: Optional[str] = Field(
        None, alias="oracle_instant_client_path"
    )
    AWS_ACCESS_KEY_ID: str
    AWS_SECRET_ACCESS_KEY: str
    AWS_BEDROCK_REGION: str
    BEDROCK_MODEL_ID: str = AppConstants.BEDROCK_DEFAULT_MODEL_ID
    AWS_SERVICE_NAME: str
    MARIADB_USER: str
    MARIADB_PASSWORD: str
    MARIADB_HOST: str
    MARIADB_PORT: int = 3306
    MARIADB_DATABASE: str
    API_BASE_URL: Optional[str] = Field(None, alias="api_base_url")
    API_USERNAME: Optional[str] = Field(None, alias="username")
    API_PASSWORD: Optional[str] = Field(None, alias="password")

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"

In [4]:
def create_boto3_session(
    settings: Settings, config: Optional[Config] = None
) -> boto3.Session:
    try:
        logger.info(
            f"Criando sessão boto3 para a região: {settings.AWS_BEDROCK_REGION}..."
        )
        session = boto3.Session(
            region_name=settings.AWS_BEDROCK_REGION,
            aws_access_key_id=settings.AWS_ACCESS_KEY_ID,
            aws_secret_access_key=settings.AWS_SECRET_ACCESS_KEY,
        )
        logger.info("Sessão boto3 criada com sucesso.")
        return session
    except Exception as e:
        logger.critical(f"Não foi possível criar a sessão boto3: {e}")
        raise

## geting the service ready to use
### model configuration

In [5]:

app_constants = AppConstants()
settings = Settings()

# Create a custom boto3 session

session = create_boto3_session(settings)

# Create a Bedrock model with the custom session
bedrock_model = BedrockModel(
    model_id=settings.BEDROCK_MODEL_ID,
    boto_session=session,
    streaming=False,
    temperature=app_constants.TEMPERATURE,
    top_p=app_constants.TOP_P,
    boto_client_config=Config(
        retries=app_constants.RETRIES,
        connect_timeout=app_constants.CONNECTION_TIMEOUT,
        read_timeout=app_constants.READ_TIMEOUT
    )
)

{"timestamp": "2025-08-13T08:15:18", "level": "INFO", "name": "__main__", "message": "Criando sessão boto3 para a região: us-east-1...", "filename": "244079052.py", "lineno": 5}
{"timestamp": "2025-08-13T08:15:18", "level": "INFO", "name": "__main__", "message": "Sessão boto3 criada com sucesso.", "filename": "244079052.py", "lineno": 13}


In [ ]:
# working with AWS texttract to get text from pdf
pdf_path = "/home/joao/projects/company_projects/carteirinha-api/documents/pdf_carteirinha/LO_DOCUMENTO_ANEXO_CIRURGICO.pdf"


In [6]:
from strands import Agent

agent = Agent(model=bedrock_model)

prompt = "Tell me about Amazon Bedrock."
response = agent(prompt=prompt)


# Amazon Bedrock

Amazon Bedrock is a fully managed service that provides access to a range of foundation models (FMs) through a unified API. It allows developers to build and scale generative AI applications without having to manage the underlying infrastructure.

Key features include:

- **Multiple model options**: Access to models from leading AI companies including Amazon's own Titan models, Anthropic's Claude, AI21 Labs' Jurassic, Cohere, Meta, and Stability AI
- **Customization capabilities**: Fine-tune models with your own data
- **Security and privacy**: Enterprise-grade security with private endpoints and data encryption
- **Seamless integration**: Works with other AWS services
- **Serverless experience**: Pay only for what you use with no infrastructure management

Bedrock enables various applications like content generation, summarization, chatbots, search enhancement, and code generation while keeping your data and prompts private.

In [36]:
# system prompt will be passed as a txt file
class AgentCarteirinha():
    """ This agent will use a multimodal LLM.
    The user will input a list of images on base64, and the agent will process them accordingly, based
    in its agent definition by the system prompt and the passed images.
    The output should be a well structured json"""
    def __init__(self, model: BedrockModel, system_prompt: str, data_model: PydanticBaseModel):
        self.model = model
        self.system_prompt = system_prompt
        self.agent = Agent(model=self.model, system_prompt=system_prompt)
        self.data_model = data_model

    def extract_text_from_pdf(self, pdf_bytes: bytes) -> Dict:
        """ This function will receive a pdf file in bytes format,
        and will return a structured json with the extracted data.
        We append the pdf file and our agent definition as prompt to send
        on a single request."""
        pdf_content =  {
            "document": {
                "format": "pdf",
                "name": f"pdf_blob_{len(pdf_bytes)}.pdf",
                "source": {
                    "bytes": pdf_bytes
                }
            }
        }

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Process the attached PDF according to the system prompt."},
                    pdf_content
                ]
            }
        ]
        return self.agent(messages=messages, prompt="analyze the PDF and return structured data according to the system prompt")



    def extract_data_from_images_bytes(self, list_images_bytes: List[bytes]) -> Dict:
        """ This function will receive a list of images in bytes png format,
        and will return a structured json with the extracted data.
        We append max of 20 images and our agent definition as prompt to send
        on a single request."""

        images_text_list = []
        for idx, img_bytes in enumerate(list_images_bytes[:20], start=0):
            images_text_list.append({
                "document": {
                    "format": "png",
                    "name": f"image_{idx}",
                            "source": {
                                "bytes": img_bytes
                            }
                        }
                    })

        messages = [
            {"role": "user",
             "content": images_text_list
            }
        ]

        prompt="Extract the data from the png documents provided and return it in a structured JSON format. follow the system prompt"

        return self.agent(messages=messages, prompt=prompt)
        # return self.agent.structured_output(output_model=self.data_model, prompt=images_text_list)



In [8]:
# image utils
MAX_IMAGES_PER_BLOB = 20  # Maximum number of images per blob
def converter_blob_para_imagens(blob: bytes, extensao: str) -> List[Image.Image]:
    imagens = []
    ext = extensao.lower().strip(".") if extensao else ""
    try:
        if ext == "pdf":
            with fitz.open(stream=blob, filetype="pdf") as pdf_doc:
                logger.info(f"Processando PDF com {len(pdf_doc)} página(s)...")
                for pagina in pdf_doc: # limit to 20 pages
                    if len(imagens) >= MAX_IMAGES_PER_BLOB:
                        logger.warning(f"⚠️ BLOB has reached the maximum limit of {MAX_IMAGES_PER_BLOB} images.")
                        break
                    pix = pagina.get_pixmap(matrix=fitz.Matrix(3.0, 3.0), alpha=False)
                    imagens.append(Image.open(io.BytesIO(pix.tobytes("png"))))
        elif ext in ["jpg", "jpeg", "png", "bmp"]:
            imagens.append(Image.open(io.BytesIO(blob)))
        else:
            logger.warning(f"Formato de arquivo não suportado: '{ext}'.")
    except Exception as e:
        logger.error(f"Erro ao converter BLOB para imagem (ext: .{ext}): {e}")
    return imagens


def aplicar_clahe(imagem: Image.Image) -> Image.Image:
    try:
        imagem_cv = cv2.cvtColor(np.array(imagem), cv2.COLOR_RGB2BGR)
        imagem_cinza = cv2.cvtColor(imagem_cv, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return Image.fromarray(clahe.apply(imagem_cinza))
    except Exception:
        return imagem

def imagens_para_bytes(lista_imagens: List[Image.Image], formato: str = "PNG") -> List[bytes]:
    imagens_bytes = []
    TARGET_BYTES = 3.5 * 1024 * 1024
    for img in lista_imagens:
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        for quality in range(95, 15, -10):
            buffer = io.BytesIO()
            img.save(buffer, format="png", quality=quality)
        imagens_bytes.append(buffer.getvalue())
    return imagens_bytes

## load the dataset and explore for prompt ideas

In [9]:
import sqlalchemy
data_path = "gold_carteirinha_database.sqlite"
engine = sqlalchemy.create_engine(f"sqlite:///{data_path}")
df_merged_final = pd.read_sql("SELECT * FROM carteirinha", engine)

df_merged_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 221 entries, 0 to 220
Data columns (total 37 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   CD_AVISO_CIRURGIA             221 non-null    int64 
 1   CD_DOCUMENTO_ANEXO_CIRURGICO  221 non-null    int64 
 2   CD_GUIA                       221 non-null    int64 
 3   TP_GUIA                       221 non-null    object
 4   TP_SITUACAO                   221 non-null    object
 5   LO_DOCUMENTO_ANEXO_CIRURGICO  221 non-null    object
 6   DS_EXTENSAO                   221 non-null    object
 7   DT_ANEXO                      221 non-null    object
 8   DS_DOCUMENTO_ANEXO            221 non-null    object
 9   CD_PACIENTE                   221 non-null    int64 
 10  NR_CARTEIRA                   221 non-null    object
 11  CD_CONVENIO                   221 non-null    int64 
 12  NM_EMPRESA                    0 non-null      object
 13  DT_INTEGRA          

In [10]:
# get a dataframe that groups by NOME_CONVENIO
df_grouped = df_merged_final[["NOME_CONVENIO", "NR_CARTEIRA"]].groupby("NOME_CONVENIO").agg(list).reset_index()


# geting a new df  with only the first ocurrency of NR_CARTEIRA FOR EACH NOME_CONVENIO
df_first_occurrence = df_grouped.explode("NR_CARTEIRA").drop_duplicates(subset=["NOME_CONVENIO"], keep="first").reset_index(drop=True)

# NOW lets ADD the len of NR_CARTEIRA col
df_first_occurrence["NR_CARTEIRA_LEN"] = df_first_occurrence["NR_CARTEIRA"].apply(lambda x: len(x) if isinstance(x, str) else 0)

df_first_occurrence = df_first_occurrence.sort_values(by="NR_CARTEIRA_LEN", ascending=False).reset_index(drop=True)

print(df_first_occurrence)

                    NOME_CONVENIO        NR_CARTEIRA  NR_CARTEIRA_LEN
0                  UNIMED SEGUROS  9942432512326010                17
1           SUL AMERICA DIRETO BH  88888474451670021               17
2                     SUL AMERICA  88888483405330026               17
3             STELLANTIS SAUDE MG  00010001091710017               17
4         CAIXA ECONOMICA FEDERAL   0100409841000238               16
5                    BLUE COMPANY   0000000620569300               16
6                           CASSI   1101703901120038               16
7                            IPSM   6500116540743112               16
8         POSTAL SAUDE - CORREIOS   0184140747000288               16
9                        BRADESCO    954560112208012               15
10             BRADESCO OPERADORA    954420014320014               15
11             PLAN ASSISTE - MPF     10511003380000               14
12                      CARE PLUS       090900038401               12
13              PETR

### load a blob to our agent and see the response

In [11]:
# data model
class CarteirinhaExtraida(PydanticBaseModel):
    """Define a estrutura dos dados extraídos da carteirinha."""
    cd_aviso_cirurgia: Optional[str] = Field(None, description="ID do aviso de cirurgia")
    confidence_its_carteirinha: Optional[float] = Field(None, description="Confiança na extração da carteirinha.")
    real_carteirinha_num: Optional[str] = Field(None, description="Número real da carteirinha.")
    convenio: Optional[str] = Field(None, description="Nome do convênio de saúde.")
    plano: Optional[str] = Field(None, description="Nome do plano de saúde.")
    nome_pessoa: Optional[str] = Field(None, description="Nome completo do titular ou beneficiário.")
    numero_carteirinha: Optional[str] = Field(None, description="O número de identificação da carteirinha.")

In [12]:
# get the entire row from example blob
df_merged_final.iloc[2]

CD_AVISO_CIRURGIA                                                          792062
CD_DOCUMENTO_ANEXO_CIRURGICO                                              2458205
CD_GUIA                                                                  18238003
TP_GUIA                                                                         I
TP_SITUACAO                                                                     A
LO_DOCUMENTO_ANEXO_CIRURGICO    b'%PDF-1.4\n1 0 obj\n<<\n/Title (\xfe\xff\x00D...
DS_EXTENSAO                                                                  .pdf
DT_ANEXO                                               2025-01-02 09:28:29.000000
DS_DOCUMENTO_ANEXO                                           Carteira do convênio
CD_PACIENTE                                                               1991926
NR_CARTEIRA                                                       775045002283004
CD_CONVENIO                                                                    20
NM_EMPRESA      

In [41]:
example_blob = df_merged_final.iloc[1]["LO_DOCUMENTO_ANEXO_CIRURGICO"]

images = converter_blob_para_imagens(example_blob, "pdf")
# optional: save the image 
save_path = "docs/images/"
os.makedirs(save_path, exist_ok=True)
for i, img in enumerate(images):
    img.save(f"{save_path}/image_{i}.png")

images_with_clahe = [aplicar_clahe(img) for img in images]
images_bytes_list = imagens_para_bytes(images_with_clahe)



#transform the images to png format
images_png = [img.convert("RGBA") for img in images_with_clahe]



{"timestamp": "2025-08-13T08:30:00", "level": "INFO", "name": "__main__", "message": "Processando PDF com 1 página(s)...", "filename": "708635088.py", "lineno": 9}


In [14]:
# load the query from the docs/carteirinha_agent_prompt_v3.txt
with open("docs/carteirinha_agent_prompt_v4.txt", "r", encoding="utf-8") as file:
    carteirinha_agent_prompt = file.read()


In [15]:
carteirinha_agent_prompt = carteirinha_agent_prompt.strip()


In [16]:
# same prompt but here

last_prompt = r"""
Você é um especialista em OCR e extração estruturada de dados.  
Sua tarefa é analisar uma ou mais imagens e retornar **apenas** as informações de carteirinhas de convênio de saúde no formato JSON.

IMPORTANTE: Retorne **apenas** o objeto JSON final, sem explicações, textos adicionais ou comentários.

---

ETAPA 1 — Identificar se há carteirinha de convênio
- Analise todas as imagens recebidas.
- Caso não encontre carteirinha, mas haja informações relevantes (por exemplo, em uma página web), ainda assim extraia.
- Se não encontrar nenhum campo válido, retorne os campos como `null`.

---

ETAPA 2 — Campos a extrair
1. **convenio**: Nome do convênio de saúde.  
2. **plano**: Nome ou tipo do plano (ex.: "Plano Prata", "Enfermaria").  
3. **nome_pessoa**: Nome completo do beneficiário.  
4. **numero_carteirinha**: Número da carteirinha, processado conforme as regras de limpeza e validação abaixo.

---

ETAPA 3 — Limpeza de dados e aplicação de regras especiais
- Remova todos caracteres especiais usando regex: `[^a-zA-Z0-9\s]`.
- Antes de validar o tamanho, **aplique as regras especiais da ETAPA 5**.
- Após aplicar as regras especiais, valide que `numero_carteirinha` tenha exatamente a quantidade de dígitos esperada para o convênio (ver ETAPA 4).
- Caso o convênio não esteja na lista de mapeamento, retorne `null` no campo `convenio` e `numero_carteirinha`.

---

ETAPA 4 — Lista de Mapeamento
- Primeiro identifique o convenio.
Use esta lista para identificar quantos digitos o numero de carteirinha deve ter baseada no convenio identificado.
Caso o numero relativo ao convenio encontrado seja "variavel", nao valide o tamanho.

[
("STELLANTIS SAUDE MG", 17),
("SUL AMERICA", variavel),
("CASSI", 16),
("CAIXA ECONOMICA FEDERAL", 11),
("BLUE COMPANY", 16),
("POSTAL SAUDE - CORREIOS", 16),
("IPSM", 16),
("UNIMED SEGUROS", 16),
("BRADESCO", 15),
("BRADESCO OPERADORA", 15),
("PLAN ASSISTE - MPF", 14),
("CARE PLUS", 12),
("PETROBRAS - REGAP", 12),
("VALE - AMS", 12),
("FUNDAFFEMG", 12),
("CEMIG SAUDE", variavel),
("VALE - PASA", 10),
("AMIL", 9),
("AMIL VM (ANTIGA GOLDEN CROSS)", 9),
("COPASS", 8),
("SPA SAUDE", 5)
]

---

ETAPA 5 — Regras Especiais
1. **CEMIG SAUDE**  
   - Se houver dois números, use o da matrícula do beneficiário (não a matrícula antiga).
2. **SUL AMERICA**  
  - Se o número identificado tiver 20 dígitos, **NUNCA** retorne os 3 primeiros dígitos originais.  
  - Remova exatamente os 3 primeiros dígitos e mantenha apenas os 17 últimos.  
  - O valor final retornado no campo `numero_carteirinha` deve ter **exatamente 17 dígitos**.  
  - Se, após cortar, não tiver 17 dígitos, retorne `null` neste campo.  

---

ETAPA 6 — Resposta Final
O numero de carteirinha deve ser retornado sempre com a quantidade de digitos esperada, baseada na lista de mapeamento.
Formato obrigatório:
```json
{
  "convenio": "string",
  "plano": "string",
  "nome_pessoa": "string",
  "numero_carteirinha": "string"
}

Todos os campos que não puderem ser identificados devem ter valor null.
Não inclua texto, explicações ou comentários fora do JSON.


"""

In [37]:
# instantiate our agent

carteirinha_agent_prompt = last_prompt

carteirinha_agent = AgentCarteirinha(model=bedrock_model, system_prompt=carteirinha_agent_prompt, data_model=CarteirinhaExtraida)

In [38]:
# send the image bytes to agent and get a response
response = carteirinha_agent.extract_text_from_pdf(pdf_bytes=example_blob)

```json
{
  "convenio": "BRADESCO",
  "plano": "SAUDE NACIONAL FLEX E CA COPAR",
  "nome_pessoa": "MARIA APARECIDA SILVA",
  "numero_carteirinha": "123456789012345"
}
```

In [39]:
images_bytes_list

[b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\t\xde\x00\x00\x06\xf9\x08\x00\x00\x00\x00u8\xdfZ\x00\x01\x00\x00IDATx\x9c\xec\xddu|\x14w\xfe\xc7\xf1W\x081\x02Ipw\xd7B\x0b\xa5H\x8bS\xa3\xeenw\xed\xf5z\x95\xbb^\xdd{\xf5\xfb]\xbd\xd7\xabB]\xa1-\xb4\x14www\x97\x00\x01bD7\xd9\xe4\xf7\xc7\xac\xcc\xec\xce&\x9bBH2\xbc\x9f\x8f\x07\x0fvg\xbe\xf3\x95\xd9\x9d\xd9O\xbe3\xdf\xefD\x14#""""\xceQ\xad\xa2+ """"\'\x92\xc2;\x11\x11\x11\x11GQx\'"""\xe2(\n\xefDDDD\x1cE\xe1\x9d\x88\x88\x88\x88\xa3(\xbc\x13\x11\x11\x11q\x14\x85w""""\x8e\xa2\xf0NDDD\xc4Q\x14\xde\x89\x88\x88\x888\x8a\xc2;\x11\x11\x11\x11GQx\'"""\xe2(\n\xefDDDD\x1cE\xe1\x9d\x88\x88\x88\x88\xa3(\xbc\x13\x11\x11\x11q\x14\x85w""""\x8e\xa2\xf0NDDD\xc4Q\x14\xde\x89\x88\x88\x888\x8a\xc2;\x11\x11\x11\x11GQx\'"""\xe2(\n\xefDDDD\x1cE\xe1\x9d\x88\x88\x88\x88\xa3(\xbc\x13\x11\x11\x11q\x14\x85w""""\x8e\xa2\xf0NDDD\xc4Q\x14\xde\x89\x88\x88\x888\x8a\xc2;\x11\x11\x11\x11GQx\'"""\xe2(\n\xefDDDD\x1cE\xe1\x9d\x88\x88\x88\x88\xa3(\xbc\x13\x11\x11\x11q\x14\x85w""""\

In [40]:
# extract data from image bytes

response2 = carteirinha_agent.extract_data_from_images_bytes(list_images_bytes=images_bytes_list)

```json
{
  "convenio": null,
  "plano": null,
  "nome_pessoa": null,
  "numero_carteirinha": null
}
```

## funcionou com imagens e com pdf como blob. testar com as duas approachs. pdf párece ser muito melhor

In [ ]:
# run for more 10 images and analize.